# Notebook 04 — Governance, Model Card & Deployment Handoff
**Master Playbook Section 11, 17.4, 19.1 (Phases 5, 6)**

Fill in every `REAL_*` variable below with the actual values Notebooks
02 and 03 printed — this notebook does not compute new numbers, it
assembles the ones you already have into the Model Card and points you at
the remaining real steps (deployment + governance sign-off).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../"))
import model_card_generator as mcg

# ---- Fill these in with your REAL results from Notebooks 02 and 03 ----
REAL_CHAMPION_NAME = None            # e.g. "XGBoost"
REAL_MODEL_VERSION = "v1.0.0"
REAL_TRAINING_DATA_SNAPSHOT_ID = None   # e.g. a hash or date of your creditcard.csv copy
REAL_CODE_COMMIT_HASH = None
REAL_PR_AUC_CV = None
REAL_PR_AUC_CV_CI = None             # tuple, e.g. (0.83, 0.91)
REAL_PR_AUC_TEMPORAL = None
REAL_PRECISION = None
REAL_RECALL = None
REAL_THRESHOLD = None
REAL_EXTERNAL_BENCHMARK_COMPARISON = None  # the dict mb.compare_to_external_benchmark returned

_required = [REAL_CHAMPION_NAME, REAL_TRAINING_DATA_SNAPSHOT_ID, REAL_CODE_COMMIT_HASH,
             REAL_PR_AUC_CV, REAL_PR_AUC_CV_CI, REAL_PR_AUC_TEMPORAL, REAL_PRECISION,
             REAL_RECALL, REAL_THRESHOLD, REAL_EXTERNAL_BENCHMARK_COMPARISON]
if any(v is None for v in _required):
    raise ValueError("Fill in every REAL_* variable above with your actual Notebook 02/03 results before running.")

## Assemble the real Model Card (Section 11)

In [ ]:
card = mcg.ModelCardData(
    model_name="fraud-detection-champion",
    version=REAL_MODEL_VERSION,
    training_data_snapshot_id=REAL_TRAINING_DATA_SNAPSHOT_ID,
    code_commit_hash=REAL_CODE_COMMIT_HASH,
    intended_use="Real-time transaction-level fraud probability scoring for card-not-present transactions.",
    out_of_scope_uses=["Any decision affecting protected-attribute fairness cannot be validated on this "
                        "PCA-anonymized dataset — do not present this model as fairness-audited."],
    training_data_provenance="Worldline / Machine Learning Group (ULB) Credit Card Fraud Detection dataset, "
                              "September 2013 European cardholders.",
    known_limitations=["PCA-anonymized features preclude protected-attribute fairness testing",
                        "Single-node evaluation scale", "48-hour dataset window used as a drift-monitoring proxy"],
    pr_auc_cv=REAL_PR_AUC_CV,
    pr_auc_cv_bootstrap_ci=REAL_PR_AUC_CV_CI,
    pr_auc_temporal_split=REAL_PR_AUC_TEMPORAL,
    precision_at_threshold=REAL_PRECISION,
    recall_at_threshold=REAL_RECALL,
    operating_threshold=REAL_THRESHOLD,
    external_benchmark_comparison=REAL_EXTERNAL_BENCHMARK_COMPARISON,
)

with open("model_card.md", "w") as f:
    f.write(mcg.render_markdown(card))
with open("model_card.html", "w") as f:
    f.write(mcg.render_html(card))
print("Wrote model_card.md and model_card.html from your real results.")

## Governance sign-off (Section 19.1, Phase 6)

Open `governance_signoff_template.md` and fill in every row using the real
results from Notebooks 01–03 and this notebook's Model Card. This is a
document you fill in by hand (or with a co-reviewer) — running more code
cannot substitute for a real, dated sign-off.

## Deployment handoff (Section 17.4, Phase 5)

1. Copy `champion_model.pkl` into `deployment/model/champion_model.pkl`.
2. Set `MODEL_VERSION` and `DECISION_THRESHOLD` in `deployment/docker-compose.yml` to your real values above.
3. Run `docker compose up` from the `deployment/` folder.
4. Send a real request and record the REAL measured latency (not the local proxy below) as your Phase 5 evidence:

```
curl -X POST http://localhost:8000/score -H "Content-Type: application/json" \
     -d '{"features": {"V1": ..., "V2": ..., ..., "Amount": 149.62}}'
```


## Local latency proxy (NOT the real containerized p99 — a first, honest local signal only)

In [ ]:
import time
import numpy as np

sample = df[feature_cols].iloc[:200].values if 'df' in dir() else None
if sample is not None:
    latencies_ms = []
    for row in sample:
        t0 = time.perf_counter()
        champion_model.predict_proba(row.reshape(1, -1))
        latencies_ms.append((time.perf_counter() - t0) * 1000)
    p99 = np.percentile(latencies_ms, 99)
    print(f"LOCAL proxy p99 latency over {len(sample)} real rows: {p99:.3f} ms "
          f"— this is NOT the containerized measurement Phase 5 requires, only a sanity check.")
else:
    print("Run this from a session where `df` and `champion_model` are already loaded (e.g. append to Notebook 03).")

## Your real conclusions (fill in AFTER completing the steps above)

- **Model card written and reviewed:** yes / no
- **Governance sign-off status:** _[which tiers signed, date]_
- **Real containerized p99 latency:** _[value, from the curl/deployment step — NOT the local proxy]_
- **Overall Phase 1/3/5/6 status:** _[your honest assessment]_
